In [328]:
from datetime import date, datetime, timedelta
import pandas as pd
import weather_forecast as wf
from mlflow import mlflow, MlflowClient
import pickle
import xgboost  as xgb

In [335]:
#current_dateTime = datetime.now()
#current_dateTime= datetime(2025,8,3,15,30)
current_dateTime= datetime(2025,8,5,10,0)
current_date=current_dateTime.strftime("%Y-%m-%d")
current_hour=int(current_dateTime.strftime("%H"))
current_minute=int(current_dateTime.strftime("%M"))
weather_hour = current_hour if current_minute<30 else (current_hour+1)
weather_time=str(weather_hour)+":00:00"
current_date, current_hour

('2025-08-05', 10)

In [336]:
current_date, current_minute, weather_time, 

('2025-08-05', 0, '10:00:00')

In [337]:
latitude=45.73253954365124
longitude=9.714445021009563
timezone="Europe/Berlin"

In [338]:
df_hourly=wf.get_weather_data(start_date=current_date, end_date=current_date,latitude=latitude,longitude=longitude, timezone=timezone)
df_hourly

,timestamp,weather_code,cloud_cover,temperature
0,2025-08-04 23:00:00,1.0,27.0,16.200001
1,2025-08-05 00:00:00,1.0,23.0,15.850000
2,2025-08-05 01:00:00,0.0,15.0,16.750000
3,2025-08-05 02:00:00,0.0,4.0,16.900000
4,2025-08-05 03:00:00,0.0,0.0,16.799999
5,2025-08-05 04:00:00,0.0,0.0,16.250000
6,2025-08-05 05:00:00,0.0,0.0,16.400000
7,2025-08-05 06:00:00,0.0,0.0,19.000000
8,2025-08-05 07:00:00,0.0,0.0,21.850000
9,2025-08-05 08:00:00,0.0,0.0,22.650000


In [339]:
df_eval=pd.DataFrame()
weather_fields=["weather_code","cloud_cover","temperature"]
df_eval["time"]=df_hourly.timestamp.dt.time
df_eval["month"] = current_dateTime.month
df_eval[weather_fields] = df_hourly[weather_fields]
df_eval

,time,month,weather_code,cloud_cover,temperature
0,23:00:00,8,1.0,27.0,16.200001
1,00:00:00,8,1.0,23.0,15.850000
2,01:00:00,8,0.0,15.0,16.750000
3,02:00:00,8,0.0,4.0,16.900000
4,03:00:00,8,0.0,0.0,16.799999
5,04:00:00,8,0.0,0.0,16.250000
6,05:00:00,8,0.0,0.0,16.400000
7,06:00:00,8,0.0,0.0,19.000000
8,07:00:00,8,0.0,0.0,21.850000
9,08:00:00,8,0.0,0.0,22.650000


In [340]:

df_eval = df_eval[ ( (df_eval.time.astype(str)>=weather_time) & (df_eval.time.astype(str)<'22:00:00') ) ]
df_eval, weather_time


(        time  month  weather_code  cloud_cover  temperature
 11  10:00:00      8           2.0         62.0    24.600000
 12  11:00:00      8           0.0          5.0    25.350000
 13  12:00:00      8           0.0          1.0    25.750000
 14  13:00:00      8           0.0          6.0    26.100000
 15  14:00:00      8           0.0          5.0    26.250000
 16  15:00:00      8           0.0          8.0    26.250000
 17  16:00:00      8           0.0          6.0    26.000000
 18  17:00:00      8           0.0         15.0    25.450001
 19  18:00:00      8           0.0          4.0    23.750000
 20  19:00:00      8           0.0          7.0    22.049999
 21  20:00:00      8           0.0          2.0    21.000000
 22  21:00:00      8           0.0          3.0    20.299999,
 '10:00:00')

In [341]:
myrow = df_eval.iloc[0]
type(myrow["time"])
dt = datetime.combine(date.today(), myrow["time"]) + timedelta(minutes=30)
print (dt.time())
#myrow["time"] = myrow["time"] + timedelta(minutes=30)
#myrow

10:30:00


In [342]:
df_eval

,time,month,weather_code,cloud_cover,temperature
11,10:00:00,8,2.0,62.0,24.600000
12,11:00:00,8,0.0,5.0,25.350000
13,12:00:00,8,0.0,1.0,25.750000
14,13:00:00,8,0.0,6.0,26.100000
15,14:00:00,8,0.0,5.0,26.250000
16,15:00:00,8,0.0,8.0,26.250000
17,16:00:00,8,0.0,6.0,26.000000
18,17:00:00,8,0.0,15.0,25.450001
19,18:00:00,8,0.0,4.0,23.750000
20,19:00:00,8,0.0,7.0,22.049999


In [343]:
# Replicate weather data at half-hours to match model data
df_eval_30 = df_eval.copy()
today = date.today()
for idx, row in df_eval_30.iterrows():
    dt = datetime.combine(today, row["time"]) + timedelta(minutes=30)
    df_eval_30.at[idx,"time"] = dt.time()

df_eval = pd.concat([df_eval, df_eval_30]).sort_values('time')
df_eval


,time,month,weather_code,cloud_cover,temperature
11,10:00:00,8,2.0,62.0,24.600000
11,10:30:00,8,2.0,62.0,24.600000
12,11:00:00,8,0.0,5.0,25.350000
12,11:30:00,8,0.0,5.0,25.350000
13,12:00:00,8,0.0,1.0,25.750000
13,12:30:00,8,0.0,1.0,25.750000
14,13:00:00,8,0.0,6.0,26.100000
14,13:30:00,8,0.0,6.0,26.100000
15,14:00:00,8,0.0,5.0,26.250000
15,14:30:00,8,0.0,5.0,26.250000


In [344]:
mlflow_tracking_uri = "http://localhost:5000"
experiment_name="solar-surplus-experiment"

mlflow.set_tracking_uri(mlflow_tracking_uri)
mlflow.set_experiment(experiment_name)

runs = mlflow.search_runs(filter_string="status='FINISHED'")
#df_runs = client.search_runs(experiment_ids=experiment_name,filter_string="status='FINISHED'")
#df_runs = client.search_runs(experiment_ids=experiment_name)
runs


,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.rmse_test,metrics.validation-rmse,params.seed,params.min_child_weight,...,params.num_boost_round,params.early_stopping_rounds,params.maximize,params.custom_metric,params.verbose_eval,tags.mlflow.user,tags.mlflow.source.type,tags.mlflow.source.name,tags.mlflow.runName,tags.mlflow.source.git.commit
0,db6fd747ff924f55a1c18cfacfee6820,1,FINISHED,mlflow-artifacts:/1/db6fd747ff924f55a1c18cfacf...,2026-02-05 10:24:05.976000+00:00,2026-02-05 10:24:12.820000+00:00,318.186206,NaN,42,11.057750768573072,...,None,None,None,None,None,default,LOCAL,/home/airflow/.local/bin/airflow,mercurial-shad-20,None
1,63d81915bce24eaa8541b17d457db7f1,1,FINISHED,mlflow-artifacts:/1/63d81915bce24eaa8541b17d45...,2026-02-05 10:11:44.034000+00:00,2026-02-05 10:11:51.042000+00:00,319.763349,NaN,42,11.057750768573072,...,None,None,None,None,None,default,LOCAL,/home/airflow/.local/bin/airflow,brawny-cow-514,None
2,0685a4323b56417c8296ed58fbf5206c,1,FINISHED,mlflow-artifacts:/1/0685a4323b56417c8296ed58fb...,2026-01-29 16:30:56.105000+00:00,2026-01-29 16:31:03.598000+00:00,314.226975,NaN,42,11.057750768573072,...,None,None,None,None,None,calinet,LOCAL,solar-estimator-model-pipeline.py,capable-jay-900,56b827995db1dfbc541b8124b61e050cc43b9bb0
3,2e611c64f11342a99743da06e758a2f1,1,FINISHED,mlflow-artifacts:/1/2e611c64f11342a99743da06e7...,2026-01-29 16:23:26.264000+00:00,2026-01-29 16:23:51.939000+00:00,316.992495,NaN,42,11.057750768573072,...,None,None,None,None,None,calinet,LOCAL,solar-estimator-model-pipeline.py,legendary-grub-271,56b827995db1dfbc541b8124b61e050cc43b9bb0
4,89768031accd464d9befca8af73b8f88,1,FINISHED,mlflow-artifacts:/1/89768031accd464d9befca8af7...,2026-01-29 15:16:39.041000+00:00,2026-01-29 15:21:10.516000+00:00,319.034283,319.045534,42,11.057750768573072,...,1000,50,None,None,True,calinet,LOCAL,solar-estimator-model-pipeline.py,stylish-stork-851,56b827995db1dfbc541b8124b61e050cc43b9bb0


In [345]:
best_run = runs.loc[ runs['metrics.rmse_test'].idxmin() ]
best_run_id = best_run['run_id']
best_run_rmse = best_run['metrics.rmse_test']
print(f'best run_id: {best_run_id}, rmse: {best_run_rmse}')
best_run, best_run.artifact_uri

best run_id: 0685a4323b56417c8296ed58fbf5206c, rmse: 314.2269750290191


(run_id                                            0685a4323b56417c8296ed58fbf5206c
 experiment_id                                                                    1
 status                                                                    FINISHED
 artifact_uri                     mlflow-artifacts:/1/0685a4323b56417c8296ed58fb...
 start_time                                        2026-01-29 16:30:56.105000+00:00
 end_time                                          2026-01-29 16:31:03.598000+00:00
 metrics.rmse_test                                                       314.226975
 metrics.validation-rmse                                                        NaN
 params.seed                                                                     42
 params.min_child_weight                                         11.057750768573072
 params.objective                                                  reg:squarederror
 params.max_depth                                                           

In [346]:
client = MlflowClient(mlflow_tracking_uri)
 
# Download run artifact
path = client.download_artifacts(run_id=best_run_id, path="preprocessor/preprocessor.b")

with open(path, 'rb') as f_out:
   dv = pickle.load(f_out)


In [347]:
model_filter_string=f"source_run_id='{best_run_id}'"
models = mlflow.search_logged_models(filter_string=model_filter_string)
best_model_id=models.at[0,"model_id"]
best_model_id

'm-39519ca974a7412281c395e355db9252'

In [348]:
#filter_string=f"run_id='{best_run_id}'"
#model_name="solar-energy-surplus-predictor"
#model_id="m-39519ca974a7412281c395e355db9252"
#filter_string="name='solar-energy-surplus-predictor'"
#model_version=mlflow.search_model_versions(filter_string=filter_string)
#model_version=mlflow.search_model_versions()
#filter_string = f"name='{model_name}'"
#results = client.search_model_versions()
#logged_model=f"runs:/{best_run_id}/models/"
logged_model=f"models:/{best_model_id}"
#runs:/<mlflow_run_id>/run-relative/path/to/model
booster = mlflow.xgboost.load_model(logged_model)
# model_data = client.get_logged_model(logged_model)
# model = joblib.load(model_data)
best_run_id, results, booster
#logged_model

('0685a4323b56417c8296ed58fbf5206c',
 [],
 <xgboost.core.Booster at 0x726e34fc2cf0>)

In [349]:
# Set features
categorical=['time', 'month','weather_code']
numerical=[ 'cloud_cover', 'temperature' ]

df_eval[categorical] = df_eval[categorical].astype(str)

X_eval_dicts = df_eval[categorical+numerical].to_dict(orient='records')
#X_eval_dicts

for row in X_eval_dicts:
    print(row)
    X_eval = dv.transform(row)
    X_eval_dm = xgb.DMatrix(X_eval)
    y_pred = booster.predict(X_eval_dm)
    print(y_pred)   

#X_eval = dv.transform(X_eval_dicts)

# X_eval_dm = xgb.DMatrix(X_eval)

# y_pred = booster.predict(X_eval_dm)
# y_pred



{'time': '10:00:00', 'month': '8', 'weather_code': '2.0', 'cloud_cover': 62.0, 'temperature': 24.600000381469727}
[1118.9604]
{'time': '10:30:00', 'month': '8', 'weather_code': '2.0', 'cloud_cover': 62.0, 'temperature': 24.600000381469727}
[1355.9266]
{'time': '11:00:00', 'month': '8', 'weather_code': '0.0', 'cloud_cover': 5.0, 'temperature': 25.350000381469727}
[982.5672]
{'time': '11:30:00', 'month': '8', 'weather_code': '0.0', 'cloud_cover': 5.0, 'temperature': 25.350000381469727}
[914.9982]
{'time': '12:00:00', 'month': '8', 'weather_code': '0.0', 'cloud_cover': 1.0, 'temperature': 25.75}
[1363.8838]
{'time': '12:30:00', 'month': '8', 'weather_code': '0.0', 'cloud_cover': 1.0, 'temperature': 25.75}
[1063.2227]
{'time': '13:00:00', 'month': '8', 'weather_code': '0.0', 'cloud_cover': 6.0, 'temperature': 26.100000381469727}
[1084.6816]
{'time': '13:30:00', 'month': '8', 'weather_code': '0.0', 'cloud_cover': 6.0, 'temperature': 26.100000381469727}
[1178.5669]
{'time': '14:00:00', 'mont